# Transformación y enriquecimiento del dataset Titanic para AE3

## Necesidad y alcance
Un equipo académico que prepara material sobre seguridad marítima necesita describir cómo se distribuyó la supervivencia observada entre los pasajeros registrados. Se conserva el alcance descriptivo de AE2: se caracterizan asociaciones dentro del dataset, sin atribuir causalidad ni generalizar a otros accidentes.

Se usa el CSV generado en AE1 a partir de `BIT/titanic-dataset`, split `train`. Contiene las seis variables pertinentes y permite reproducir AE3 sin una descarga durante cada ejecución.

## Preguntas variables y resultados esperados

| Pregunta de negocio | Variables | Resultado esperado |
|---|---|---|
| 1. ¿Cuál fue la supervivencia total registrada? | `sobrevivio` | Conteos y tasa total. |
| 2. ¿Cómo varió según la clase? | `sobrevivio`, `clase_pasajero` | Conteos y tasa por clase. |
| 3. ¿Cómo varió según el sexo registrado? | `sobrevivio`, `sexo` | Conteos y tasa por sexo. |
| 4. ¿Cómo varió según el grupo de edad? | `sobrevivio`, `edad` | Conteos y tasa por intervalo, incluida la edad no registrada. |
| 5. ¿Cómo varió según el puerto? | `sobrevivio`, `ciudad_embarque` | Conteos y tasa por puerto, incluidos los faltantes. |

La pregunta compuesta de AE2 se divide en cinco preguntas operativas para generar una tabla reutilizable por pregunta. No cambian la necesidad ni las variables. `tarifa` se conserva y se describe, pero no es una pregunta principal porque AE2 la definió como complementaria y su unidad monetaria no está documentada.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
ACTIVITY_DIR = 'AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos'
if not (project_root / ACTIVITY_DIR).exists():
    project_root = project_root.parents[1]
sys.path.insert(0, str(project_root / ACTIVITY_DIR / 'src'))
from transformar_titanic import cargar_datos, ejecutar, localizar_raiz

## Carga y versión sin modificar
`df_original` se conserva para comparar dimensiones, tipos y nulos. El proceso crea otra copia antes de transformar.

In [2]:
project_root = localizar_raiz(project_root)
input_path = project_root / 'AE1' / 'resultados' / 'titanic_transformado.csv'
df_original = cargar_datos(input_path)
estructura_antes = pd.DataFrame({'tipo_antes': df_original.dtypes.astype(str), 'nulos_antes': df_original.isna().sum()})
display(df_original.head())
display(estructura_antes)
print(f'Dimensiones originales: {df_original.shape}')

,sobrevivio,clase_pasajero,sexo,edad,tarifa,ciudad_embarque
0,0,3,male,22.0,7.2500,Southampton
1,1,1,female,38.0,71.2833,Cherbourg
2,1,3,female,26.0,7.9250,Southampton
3,1,1,female,35.0,53.1000,Southampton
4,0,3,male,35.0,8.0500,Southampton


,tipo_antes,nulos_antes
sobrevivio,int64,0
clase_pasajero,int64,0
sexo,str,0
edad,float64,177
tarifa,float64,0
ciudad_embarque,str,2


Dimensiones originales: (891, 6)


## Transformación validación y enriquecimiento
Los indicadores y la clase se convierten a enteros, y edad y tarifa a valores numéricos. Se validan sus dominios y la ausencia de edades o tarifas negativas. El sexo se normaliza al español.

Los 177 faltantes de edad se conservan como `Edad no registrada` solo en `grupo_edad`; los dos puertos faltantes se representan como `No registrado`. Hay 120 coincidencias completas en las seis variables, pero no se eliminan porque no existe un identificador y la coincidencia de atributos no demuestra duplicidad.

Se añaden `estado_supervivencia`, `clase` y `grupo_edad`. Los grupos son 0 a 12, 13 a 17, 18 a 29, 30 a 44, 45 a 59, 60 o más y edad no registrada.

In [3]:
df_final, tablas, auditoria = ejecutar(project_root)
estructura_despues = pd.DataFrame({'tipo_despues': df_final.dtypes.astype(str), 'nulos_despues': df_final.isna().sum()})
comparacion = estructura_antes.join(estructura_despues, how='outer').fillna('Variable incorporada')
display(comparacion)
display(pd.DataFrame([auditoria]))
print(f'Dimensiones finales: {df_final.shape}')

/tmp/ipykernel_2570/1875237264.py:3: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  comparacion = estructura_antes.join(estructura_despues, how='outer').fillna('Variable incorporada')


,tipo_antes,nulos_antes,tipo_despues,nulos_despues
ciudad_embarque,str,2.0,string,0
clase,Variable incorporada,Variable incorporada,str,0
clase_pasajero,int64,0.0,int8,0
edad,float64,177.0,float64,177
estado_supervivencia,Variable incorporada,Variable incorporada,str,0
grupo_edad,Variable incorporada,Variable incorporada,string,0
sexo,str,0.0,str,0
sobrevivio,int64,0.0,int8,0
tarifa,float64,0.0,float64,0


,filas_antes,filas_despues,columnas_antes,columnas_despues,edades_no_registradas,puertos_no_registrados,coincidencias_en_seis_variables,filas_eliminadas,variables_incorporadas
0,891,891,6,9,177,2,120,0,3


Dimensiones finales: (891, 9)


## Resultados por pregunta
Cada tabla incluye pasajeros, sobrevivientes, no sobrevivientes, tasa de supervivencia y participación sobre los 891 registros.

In [4]:
for nombre, tabla in tablas.items():
    print(nombre)
    display(tabla)

pregunta_1_supervivencia_total


,grupo,pasajeros,sobrevivientes,no_sobrevivientes,tasa_supervivencia_pct,participacion_pasajeros_pct
0,Total,891,342,549,38.38,100.0


pregunta_2_supervivencia_por_clase


,clase,pasajeros,sobrevivientes,no_sobrevivientes,tasa_supervivencia_pct,participacion_pasajeros_pct
0,Primera,216,136,80,62.96,24.24
1,Segunda,184,87,97,47.28,20.65
2,Tercera,491,119,372,24.24,55.11


pregunta_3_supervivencia_por_sexo


,sexo,pasajeros,sobrevivientes,no_sobrevivientes,tasa_supervivencia_pct,participacion_pasajeros_pct
0,Hombre,577,109,468,18.89,64.76
1,Mujer,314,233,81,74.20,35.24


pregunta_4_supervivencia_por_grupo_edad


,grupo_edad,pasajeros,sobrevivientes,no_sobrevivientes,tasa_supervivencia_pct,participacion_pasajeros_pct
0,0 a 12,69,40,29,57.97,7.74
1,13 a 17,44,21,23,47.73,4.94
2,18 a 29,271,95,176,35.06,30.42
3,30 a 44,215,91,124,42.33,24.13
4,45 a 59,89,36,53,40.45,9.99
5,60 o más,26,7,19,26.92,2.92
6,Edad no registrada,177,52,125,29.38,19.87


pregunta_5_supervivencia_por_puerto


,puerto_embarque,pasajeros,sobrevivientes,no_sobrevivientes,tasa_supervivencia_pct,participacion_pasajeros_pct
0,Cherbourg,168,93,75,55.36,18.86
1,No registrado,2,2,0,100.00,0.22
2,Queenstown,77,30,47,38.96,8.64
3,Southampton,644,217,427,33.70,72.28


## Respuestas e interpretación

1. **Total.** Sobrevivieron 342 de 891 pasajeros, una tasa de 38,38 %.
2. **Clase.** Primera clase registró 62,96 %, segunda 47,28 % y tercera 24,24 %. Tercera clase reunió 491 pasajeros. La diferencia no prueba causalidad.
3. **Sexo.** La tasa fue 74,20 % entre 314 mujeres y 18,89 % entre 577 hombres. La comparación no controla otras variables.
4. **Edad.** El grupo de 0 a 12 años tuvo la tasa más alta entre los intervalos con edad (57,97 %). Los 177 registros sin edad representan 19,87 % y se informan aparte.
5. **Puerto.** Cherbourg registró 55,36 %, Queenstown 38,96 % y Southampton 33,70 %. Los dos registros sin puerto tienen 100 %, pero no forman un grupo estable para interpretar.

Las tablas quedan preparadas para AE4. Limitan la respuesta la procedencia original no documentada en la ficha usada en AE2, la falta de identificador único, los faltantes y el alcance descriptivo.

## Exportación y controles finales
La ejecución reemplaza automáticamente el dataset preparado, cinco CSV y el TXT. Cada tabla debe sumar 891 pasajeros y 342 sobrevivientes.

In [5]:
archivos = [project_root / ACTIVITY_DIR / 'datos' / 'titanic_preparado.csv']
archivos += sorted((project_root / ACTIVITY_DIR / 'resultados').glob('*.csv'))
archivos += [project_root / ACTIVITY_DIR / 'resultados' / 'resumen_dataframe_ae3.txt']
for archivo in archivos:
    assert archivo.exists() and archivo.stat().st_size > 0
    print(archivo.relative_to(project_root))
print('Validación final superada.')

AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos/datos/titanic_preparado.csv
AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos/resultados/pregunta_1_supervivencia_total.csv
AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos/resultados/pregunta_2_supervivencia_por_clase.csv
AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos/resultados/pregunta_3_supervivencia_por_sexo.csv
AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos/resultados/pregunta_4_supervivencia_por_grupo_edad.csv
AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos/resultados/pregunta_5_supervivencia_por_puerto.csv
AE3 Proyecto integrador - Transformación y Enriquecimiento de Datos/resultados/resumen_dataframe_ae3.txt
Validación final superada.
